# Experiment: `derived_8.4-formal-eval-2.1-ece-v3` — Formal Statistical Evaluation on In-Situ ECE Sensors (v3 Split & Salvaged Router)

Publication-oriented formal statistical evaluation of spatio-temporal soil moisture models on the **`derived_8.4_ece_v3`** dataset split (5 newly deployed in-situ stations in Western Washington: Bellevue Botanical Garden and Renton, WA; 150 rows across 2026-07-20 to 2026-08-19, native-NaN SMAP).

### Key Protocol & Evaluation Updates in v2.1:
1. **Canonical v3 Dataset:** Evaluated strictly on `derived_8.4_ece_v3` with native-NaN SMAP values reflecting real-world field deployment conditions.
2. **Missingness-Aware Router Salvage:** Deploys the availability gate router fix ($\tau = 0.10$), dynamically falling back to SMAP-free `Univariate_G_API_k2` router when SMAP blocks are missing or out-of-distribution.
3. **RMSE as Primary Evaluation Metric:** Due to the short 30-day late-summer observation window and severe ground-truth target variance compression ($\sigma_y \in [0.003, 0.008]\text{ m}^3/\text{m}^3$ at 4 of 5 stations), $R^2$ is heavily distorted by tiny residual denominators ($R^2 = 1 - \text{MSE}/\text{Var}(y)$). Consequently, models are **ranked primarily by RMSE (ascending, lower is better)**, with $R^2$, MAE, Bias, ubRMSE, and Pearson correlation ($r$) reported alongside.
4. **Trend Directionality via Pearson Correlation:** Reports Pearson $r$ across pooled and per-station evaluations to quantify whether model predictions faithfully track ground-truth dry-down curves.
5. **Time Series Visualizations (Strict $\le 5$ Lines per Chart):**
   - **Chart Suite 1 (Architecture Showdown, NO per-regime deltas):** Observed Ground Truth vs `Clustering_V0_Full_k2 c0=0,c1=0`, `Clustering_Backbone54_k2 c0=0,c1=0`, `Global_Single_54`, `Trained_Gating_k2 c0=0,c1=0` (identical 54 global features; no delta additions).
   - **Chart Suite 2 (Regime Benchmark Showdown):** Observed Ground Truth vs 4 zero-delta regime models (`Clustering_V0_Full_k2 c0=0, c1=0`, `Univariate_G_API_k2 c0=0, c1=0`, `Clustering_Dynamic_k2 c0=0, c1=0`, `Seasonal_Binary_k2 c0=0, c1=0`).

All tables below are copied verbatim from the stdout of this executed notebook.

In [1]:
"""Setup: load configs, data splits, and statistical reporting tools."""
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from scipy.stats import pearsonr
from sklearn.cluster import KMeans
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

EXP_DIR = Path(".").resolve()
PROJECT_ROOT = (EXP_DIR.parents[2]).resolve()
if str(EXP_DIR) not in sys.path:
    sys.path.insert(0, str(EXP_DIR))

import eval_formal.plots as plots
import eval_formal.stats as st
from eval_formal.configs import config_frame, load_pinned_configs
from eval_formal.data import load_experiment_data

with open(EXP_DIR / "config.yaml", encoding="utf-8") as f:
    config = yaml.safe_load(f)

data = load_experiment_data(PROJECT_ROOT, config)
configurations = load_pinned_configs(data, config)
cfg = config_frame(configurations)

METRICS = ["rmse", "r2", "mae", "bias", "pearson"]

print(f"[Setup] WA TrainVal={len(data.trainval)} WA Test={len(data.test)}")
print(f"[Setup] ECE 5 Stations (v3): All={len(data.ece_all)} (Train={len(data.ece_train)}, Val={len(data.ece_val)}, Test={len(data.ece_test)})")
print(f"[Setup] Pinned {len(cfg)} configurations, 30 seeds")

[Setup] WA TrainVal=14608 WA Test=6620
[Setup] ECE 5 Stations (v3): All=150 (Train=0, Val=0, Test=150)
[Setup] Pinned 20 configurations, 30 seeds


# Configurations (20)

14 requested configurations (test-selected deltas pinned from `derived_8.4-eval-1.1`, none = c0=c1=0)
plus 6 validation-selected winners (`val_selected_deltas.json`):

In [2]:
"""Display pinned configurations table."""
cfg_table = cfg[["config_id", "strategy_name", "delta_source", "cluster_0_count",
                 "cluster_1_count", "n_global_features", "n_add0", "n_add1"]]
print(cfg_table.to_markdown(index=False))

| config_id                            | strategy_name            | delta_source   |   cluster_0_count |   cluster_1_count |   n_global_features |   n_add0 |   n_add1 |
|:-------------------------------------|:-------------------------|:---------------|------------------:|------------------:|--------------------:|---------:|---------:|
| Clustering_V0_Full_k2_c0_0_c1_10     | Clustering_V0_Full_k2    | test           |                 0 |                10 |                  54 |        0 |       10 |
| Clustering_V0_Full_k2_c0_0_c1_0      | Clustering_V0_Full_k2    | none           |                 0 |                 0 |                  54 |        0 |        0 |
| Clustering_Backbone54_k2_c0_10_c1_10 | Clustering_Backbone54_k2 | test           |                10 |                10 |                  54 |       10 |       10 |
| Clustering_Backbone54_k2_c0_0_c1_0   | Clustering_Backbone54_k2 | none           |                 0 |                 0 |                  54 |        0

# Temporal results (Washington test set, 2023–2025, 30 seeds)

Evaluation on the frozen Washington state test split (2023–2025, 6,620 rows, 7 stations) across 30 seeds.
Replicates `derived_8.4-formal-eval-1.0` and `2.0` exactly without redundant retraining.

In [3]:
"""Temporal seed-level summary tables (RMSE, R2, Pearson, MAE, bias)."""
temporal = pd.read_csv(EXP_DIR / "temporal_seed_summary.csv")

temporal_rows = []
for cid in cfg["config_id"]:
    sub = temporal[temporal["config_id"] == cid]
    label = cfg.loc[cfg["config_id"] == cid, "config_label"].iloc[0]
    source = cfg.loc[cfg["config_id"] == cid, "delta_source"].iloc[0]
    s_rmse = st.seed_summary(sub["rmse"])
    s_r2 = st.seed_summary(sub["r2"])
    s_mae = st.seed_summary(sub["mae"])
    s_bias = st.seed_summary(sub["bias"])
    s_p = st.seed_summary(sub["pearson"]) if "pearson" in sub and sub["pearson"].notna().any() else {"mean": float("nan"), "std": float("nan"), "median": float("nan")}
    
    temporal_rows.append({
        "config_label": label,
        "delta_source": source,
        "n_seeds": int(s_rmse["n"]),
        "RMSE mean ± std": f"{s_rmse['mean']:.5f} ± {s_rmse['std']:.5f}",
        "RMSE median": s_rmse["median"],
        "R² mean ± std": f"{s_r2['mean']:.4f} ± {s_r2['std']:.4f}",
        "R² median": s_r2["median"],
        "MAE median": s_mae["median"],
        "BIAS median": s_bias["median"],
        "Pearson r median": s_p["median"],
        "rmse_raw": s_rmse["mean"],
    })

temporal_df = pd.DataFrame(temporal_rows).sort_values("rmse_raw", ascending=True).drop(columns=["rmse_raw"])
print("### Seed-level summary — Temporal WA Test (30 seeds, ranked by RMSE ascending)")
print(temporal_df.to_markdown(index=False))

### Seed-level summary — Temporal WA Test (30 seeds, ranked by RMSE ascending)
| config_label                           | delta_source   |   n_seeds | RMSE mean ± std   |   RMSE median | R² mean ± std   |   R² median |   MAE median |   BIAS median |   Pearson r median |
|:---------------------------------------|:---------------|----------:|:------------------|--------------:|:----------------|------------:|-------------:|--------------:|-------------------:|
| Clustering_V0_Full_k2  c0=0, c1=10     | test           |        30 | 0.04409 ± 0.00016 |     0.0440792 | 0.8126 ± 0.0013 |    0.812762 |    0.0339159 |    0.00658228 |           0.904411 |
| Clustering_V0_Full_k2  c0=0, c1=0      | none           |        30 | 0.04419 ± 0.00016 |     0.0441771 | 0.8118 ± 0.0014 |    0.811928 |    0.0339656 |    0.0059814  |           0.903653 |
| Clustering_Backbone54_k2  c0=0, c1=0   | none           |        30 | 0.04420 ± 0.00016 |     0.0441885 | 0.8117 ± 0.0014 |    0.811832 |    0.0339709 

# Temporal focused pairwise comparisons

Pairwise difference tests for the pre-specified comparison family on temporal RMSE and R² (mean difference,
paired t-test, Wilcoxon signed-rank, % seeds where A beats B, Benjamini–Hochberg FDR q-value).

In [4]:
"""Focused pairwise tests on temporal RMSE and R2."""
family = [
    ("Clustering_V0_Full_k2_c0_0_c1_10", "Trained_Gating_k2_c0_5_c1_10"),
    ("Clustering_V0_Full_k2_c0_0_c1_0", "Trained_Gating_k2_c0_5_c1_10"),
    ("Clustering_Backbone54_k2_c0_0_c1_0", "Trained_Gating_k2_c0_5_c1_10"),
    ("Clustering_V0_Full_k2_c0_0_c1_10", "Global_Single_54"),
    ("Clustering_V0_Full_k2_c0_0_c1_0", "Global_Single_54"),
    ("Clustering_Backbone54_k2_c0_10_c1_10", "Global_Single_54"),
    ("Clustering_Backbone54_k2_c0_0_c1_0", "Global_Single_54"),
    ("Clustering_V0_Full_k2_c0_0_c1_10", "Baseline_V0_50"),
    ("Clustering_Backbone54_k2_c0_10_c1_10", "Baseline_V0_50"),
    ("Clustering_V0_Full_k2_c0_0_c1_10", "Clustering_Backbone54_k2_c0_10_c1_10"),
    ("Clustering_Dynamic_k2_c0_10_c1_0", "Global_Single_54"),
    ("Univariate_G_API_k2_c0_10_c1_0", "Global_Single_54"),
    ("Seasonal_Binary_k2_c0_0_c1_5", "Global_Single_54"),
    ("Trained_Gating_k2_c0_5_c1_10", "Global_Single_54"),
    ("Global_Single_54", "Baseline_V0_50"),
]

def paired_test_temporal(df, a, b, metric):
    pa = df[df["config_id"] == a].set_index("seed")[metric].sort_index()
    pb = df[df["config_id"] == b].set_index("seed")[metric].sort_index()
    common = pa.index.intersection(pb.index)
    r = st.paired_test(pa.loc[common], pb.loc[common])
    # For RMSE, A wins if diff < 0; for R2, A wins if diff > 0
    if metric == "rmse":
        wins_a = float(np.mean(pa.loc[common] < pb.loc[common])) * 100.0
    else:
        wins_a = float(r["pct_A_better"])
    return {
        "A": a, "B": b, "metric": metric.upper(), "n_seeds": int(r["n"]),
        "mean_diff": r["mean_diff"], "ci": f"[{r['ci_low']:.5f}, {r['ci_high']:.5f}]",
        "t_p": r["t_p"], "wilcoxon_p": r["wilcoxon_p"],
        "pct_A_better": wins_a,
    }

temp_pairs_rmse = [paired_test_temporal(temporal, a, b, "rmse") for a, b in family]
df_temp_pairwise = pd.DataFrame(temp_pairs_rmse)
df_temp_pairwise["q_bh"] = st.bh_fdr(df_temp_pairwise["t_p"].to_numpy())
df_temp_pairwise.to_csv(EXP_DIR / "temporal_pairwise_focused.csv", index=False)
print("Focused Temporal RMSE comparisons (A vs B, negative diff favors A):")
print(df_temp_pairwise.to_markdown(index=False, floatfmt=".5f"))

Focused Temporal RMSE comparisons (A vs B, negative diff favors A):
| A                                    | B                                    | metric   |   n_seeds |   mean_diff | ci                   |     t_p |   wilcoxon_p |   pct_A_better |    q_bh |
|:-------------------------------------|:-------------------------------------|:---------|----------:|------------:|:---------------------|--------:|-------------:|---------------:|--------:|
| Clustering_V0_Full_k2_c0_0_c1_10     | Trained_Gating_k2_c0_5_c1_10         | RMSE     |        30 |    -0.00955 | [-0.00961, -0.00948] | 0.00000 |      0.00000 |      100.00000 | 0.00000 |
| Clustering_V0_Full_k2_c0_0_c1_0      | Trained_Gating_k2_c0_5_c1_10         | RMSE     |        30 |    -0.00945 | [-0.00951, -0.00939] | 0.00000 |      0.00000 |      100.00000 | 0.00000 |
| Clustering_Backbone54_k2_c0_0_c1_0   | Trained_Gating_k2_c0_5_c1_10         | RMSE     |        30 |    -0.00944 | [-0.00950, -0.00938] | 0.00000 |      0.00000 |

# Temporal sample-level cluster bootstrap (Washington test set)

Paired cluster bootstrap over (station, month) blocks across the 7 Washington test stations (126 blocks; seed-42 fits).

In [5]:
"""Sample-level block cluster bootstrap on Washington test set."""
test_df = data.test
blocks = (test_df["station_id"].astype(str) + "_" + test_df["year"].astype(str) + "_" +
          test_df["month"].astype(str)).to_numpy()
y_true = test_df[data.target].to_numpy(dtype=float)
pred_dir = EXP_DIR / "predictions"

boot_pairs = [
    ("Clustering_V0_Full_k2_c0_0_c1_10", "Global_Single_54"),
    ("Clustering_Backbone54_k2_c0_10_c1_10", "Global_Single_54"),
    ("Clustering_V0_Full_k2_c0_0_c1_10", "Baseline_V0_50"),
    ("Clustering_Backbone54_k2_c0_10_c1_10", "Baseline_V0_50"),
    ("Clustering_V0_Full_k2_c0_0_c1_10", "Trained_Gating_k2_c0_5_c1_10"),
    ("Clustering_Backbone54_k2_c0_10_c1_10", "Trained_Gating_k2_c0_5_c1_10"),
    ("Clustering_V0_Full_k2_c0_0_c1_10", "Clustering_Backbone54_k2_c0_10_c1_10"),
    ("Global_Single_54", "Baseline_V0_50"),
]

boot_rows = []
for a, b in boot_pairs:
    f_a = pred_dir / f"{a}__s42__full_preds.npy"
    f_b = pred_dir / f"{b}__s42__full_preds.npy"
    if f_a.exists() and f_b.exists():
        pa = np.load(f_a)
        pb = np.load(f_b)
        res = st.cluster_bootstrap(y_true, pa, pb, blocks, n_resamples=2000, seed=42)
        for m in ("rmse", "r2", "bias"):
            d = res[m]["diff"]
            boot_rows.append({
                "A": a, "B": b, "metric": m.upper(),
                "diff_mean": d["mean"],
                "diff CI": f"[{d['ci_low']:.5f}, {d['ci_high']:.5f}]",
                "bootstrap_p": d["p"],
            })
boot_df = pd.DataFrame(boot_rows)
boot_df.to_csv(EXP_DIR / "temporal_bootstrap.csv", index=False)
print("Sample-level paired cluster bootstrap over (station, month) blocks (seed 42):")
print(boot_df.to_markdown(index=False, floatfmt=".5f"))

Sample-level paired cluster bootstrap over (station, month) blocks (seed 42):
| A                                    | B                                    | metric   |   diff_mean | diff CI              |   bootstrap_p |
|:-------------------------------------|:-------------------------------------|:---------|------------:|:---------------------|--------------:|
| Clustering_V0_Full_k2_c0_0_c1_10     | Global_Single_54                     | RMSE     |    -0.00403 | [-0.00604, -0.00229] |       0.00050 |
| Clustering_V0_Full_k2_c0_0_c1_10     | Global_Single_54                     | R2       |     0.03584 | [0.01997, 0.05456]   |       0.00050 |
| Clustering_V0_Full_k2_c0_0_c1_10     | Global_Single_54                     | BIAS     |    -0.00405 | [-0.00586, -0.00221] |       0.00050 |
| Clustering_Backbone54_k2_c0_10_c1_10 | Global_Single_54                     | RMSE     |    -0.00110 | [-0.00347, 0.00135]  |       0.37600 |
| Clustering_Backbone54_k2_c0_10_c1_10 | Global_Single_54 

# In-Situ ECE Spatial results (5 unseen stations, derived_8.4_ece_v3, 30 seeds)

Evaluation on the 5 newly deployed In-Situ ECE sensor stations (`derived_8.4_ece_v3`, 150 rows across 2026-07-20 to 2026-08-19)
with the missingness-aware MoE router salvage enabled. Models are ranked primarily by **RMSE** (ascending, lower is better).

In [6]:
"""Spatial seed-level summary tables (ranked primarily by RMSE ascending)."""
spatial = pd.read_csv(EXP_DIR / "spatial_seed_summary.csv")
spatial_st = pd.read_csv(EXP_DIR / "spatial_seed_station.csv")

st_med = (
    spatial_st.groupby(["config_id", "station"])[METRICS]
    .median().reset_index()
)

spat_rows = []
for cid in cfg["config_id"]:
    sub_s = spatial[spatial["config_id"] == cid]
    sub_st = st_med[st_med["config_id"] == cid]
    label = cfg.loc[cfg["config_id"] == cid, "config_label"].iloc[0]
    source = cfg.loc[cfg["config_id"] == cid, "delta_source"].iloc[0]
    strategy = cfg.loc[cfg["config_id"] == cid, "strategy_name"].iloc[0]
    
    s_rmse = st.seed_summary(sub_s["rmse"])
    s_r2 = st.seed_summary(sub_s["r2"])
    s_mae = st.seed_summary(sub_s["mae"])
    s_bias = st.seed_summary(sub_s["bias"])
    s_p = st.seed_summary(sub_s["pearson"]) if "pearson" in sub_s else {"mean": float("nan"), "std": float("nan"), "median": float("nan")}
    
    spat_rows.append({
        "config_id": cid,
        "config_label": label,
        "strategy_name": strategy,
        "delta_source": source,
        "n_seeds": int(s_rmse["n"]),
        "spatial_mean_rmse": s_rmse["mean"],
        "spatial_median_rmse": s_rmse["median"],
        "spatial_mean_r2": s_r2["mean"],
        "spatial_median_r2": s_r2["median"],
        "spatial_mean_pearson": s_p["mean"],
        "spatial_median_pearson": s_p["median"],
        "RMSE mean ± std": f"{s_rmse['mean']:.5f} ± {s_rmse['std']:.5f}",
        "RMSE median": s_rmse["median"],
        "R² mean ± std": f"{s_r2['mean']:.4f} ± {s_r2['std']:.4f}",
        "R² median": s_r2["median"],
        "MAE median": s_mae["median"],
        "BIAS median": s_bias["median"],
        "Pearson r median": s_p["median"],
        "Station Median RMSE": sub_st["rmse"].median(),
        "Station Median R²": sub_st["r2"].median(),
        "Station Median Pearson": sub_st["pearson"].median() if "pearson" in sub_st else float("nan"),
    })

spat_cfg_df = pd.DataFrame(spat_rows).sort_values("spatial_mean_rmse", ascending=True)
spat_cfg_df.to_csv(EXP_DIR / "spatial_config_summary.csv", index=False)

print("### In-Situ ECE Spatial Summary (30 seeds, ranked primarily by RMSE ascending; lower is better)")
display_cols = ["config_label", "delta_source", "n_seeds", "RMSE mean ± std", "RMSE median",
                "Station Median RMSE", "R² mean ± std", "R² median", "Pearson r median"]
print(spat_cfg_df[display_cols].to_markdown(index=False))

### In-Situ ECE Spatial Summary (30 seeds, ranked primarily by RMSE ascending; lower is better)
| config_label                           | delta_source   |   n_seeds | RMSE mean ± std   |   RMSE median |   Station Median RMSE | R² mean ± std    |   R² median |   Pearson r median |
|:---------------------------------------|:---------------|----------:|:------------------|--------------:|----------------------:|:-----------------|------------:|-------------------:|
| Trained_Gating_k2  c0=5, c1=10         | test           |        30 | 0.04807 ± 0.00026 |     0.0480615 |             0.0294934 | -0.0438 ± 0.0111 |  -0.0435056 |         0.0776675  |
| Univariate_G_API_k2  c0=10, c1=10      | val            |        30 | 0.04943 ± 0.00038 |     0.0493651 |             0.0408252 | -0.1038 ± 0.0169 |  -0.100882  |        -0.00727619 |
| Clustering_Backbone54_k2  c0=10, c1=10 | test           |        30 | 0.04956 ± 0.00065 |     0.0494558 |             0.0330146 | -0.1099 ± 0.0290 |  -0.10493

# Per-station breakdown across 5 In-Situ ECE stations (derived_8.4_ece_v3)

Analysis of station-level difficulty, RMSE distribution, and Pearson correlation across the 5 in-situ sensor deployment sites.

In [7]:
"""Station-level difficulty ranking and per-station performance matrices."""
station_ranks = []
for st_id in data.ece_stations:
    sub = st_med[st_med["station"] == st_id]
    station_ranks.append({
        "station_id": st_id,
        "n_configs": len(sub),
        "median_rmse": float(sub["rmse"].median()),
        "mean_rmse": float(sub["rmse"].mean()),
        "std_rmse": float(sub["rmse"].std()),
        "median_r2": float(sub["r2"].median()),
        "mean_r2": float(sub["r2"].mean()),
        "mean_pearson": float(sub["pearson"].mean()) if "pearson" in sub else float("nan"),
        "median_pearson": float(sub["pearson"].median()) if "pearson" in sub else float("nan"),
        "mean_bias": float(sub["bias"].mean()),
    })
station_ranks_df = pd.DataFrame(station_ranks).sort_values("median_rmse", ascending=True)
print("### In-Situ ECE Station Difficulty Ranking (ranked by median RMSE ascending)")
print(station_ranks_df.to_markdown(index=False, floatfmt=".4f"))

# Per-config x station RMSE matrix
pivot_rmse = st_med.pivot(index="config_id", columns="station", values="rmse")
pivot_rmse = pivot_rmse.reindex(spat_cfg_df["config_id"])
pivot_rmse.to_csv(EXP_DIR / "spatial_per_config_station_rmse.csv")
print()
print("### Per-Configuration × Per-Station RMSE Matrix (5 ECE stations; lower is better)")
print(pivot_rmse.to_markdown(floatfmt=".4f"))

# Per-config x station Pearson matrix
if "pearson" in st_med:
    pivot_p = st_med.pivot(index="config_id", columns="station", values="pearson")
    pivot_p = pivot_p.reindex(spat_cfg_df["config_id"])
    pivot_p.to_csv(EXP_DIR / "spatial_per_config_station_pearson.csv")
    print()
    print("### Per-Configuration × Per-Station Pearson r Matrix (5 ECE stations; higher is better)")
    print(pivot_p.to_markdown(floatfmt=".4f"))

### In-Situ ECE Station Difficulty Ranking (ranked by median RMSE ascending)
| station_id              |   n_configs |   median_rmse |   mean_rmse |   std_rmse |   median_r2 |   mean_r2 |   mean_pearson |   median_pearson |   mean_bias |
|:------------------------|------------:|--------------:|------------:|-----------:|------------:|----------:|---------------:|-----------------:|------------:|
| ECE_Renton_Garden_Shed  |          20 |        0.0195 |      0.0206 |     0.0083 |    -17.6072 |  -22.9185 |         0.4310 |           0.4792 |      0.0117 |
| ECE_BBG_Main_St         |          20 |        0.0427 |      0.0370 |     0.0110 |    -57.0718 |  -46.3952 |         0.6308 |           0.7093 |      0.0347 |
| ECE_BBG_Lost_Meadow     |          20 |        0.0475 |      0.0475 |     0.0108 |    -37.7279 |  -39.6311 |        -0.4143 |          -0.4413 |      0.0439 |
| ECE_Renton_Garden_North |          20 |        0.0721 |      0.0712 |     0.0089 |     -6.7393 |   -6.6736 |        

# Spatial focused pairwise tests (5 In-Situ ECE stations)

Pairwise difference tests on per-station medians across the 5 in-situ stations: wins "k of 5 stations" (A < B for RMSE),
two-sided binomial sign test (5/5 -> p=0.0625, 4/5 -> p=0.3750), paired t-test, Wilcoxon signed-rank test,
and Benjamini–Hochberg FDR correction.

In [8]:
"""Focused pairwise tests across the 5 ECE stations on RMSE."""
def spatial_pair(med, a, b, metric):
    ma = med[med["config_id"] == a].set_index("station")[metric].reindex(
        med["station"].unique()).dropna()
    mb = med[med["config_id"] == b].set_index("station")[metric].reindex(
        med["station"].unique()).dropna()
    common = ma.index.intersection(mb.index)
    if len(common) < 2:
        return None
    r = st.station_pair_test(ma.loc[common], mb.loc[common])
    # For RMSE: A wins if diff < 0
    if metric in ("rmse", "mae", "ubrmse"):
        wins = int(np.sum(ma.loc[common] < mb.loc[common]))
    else:
        wins = int(r["wins"])
    return {
        "A": a, "B": b, "metric": metric.upper(), "n_stations": int(r["n"]),
        "mean_diff": r["mean_diff"], "wins": f"{wins} / {len(common)}",
        "sign_p": r["sign_p"], "t_p": r["t_p"], "wilcoxon_p": r["wilcoxon_p"]
    }

spatial_pairs = []
for i in range(len(cfg)):
    for j in range(i + 1, len(cfg)):
        for m in METRICS:
            r = spatial_pair(st_med, cfg["config_id"].iloc[i], cfg["config_id"].iloc[j], m)
            if r is not None:
                spatial_pairs.append(r)
spatial_pairs = pd.DataFrame(spatial_pairs)
spatial_pairs.to_csv(EXP_DIR / "spatial_pairwise_station.csv", index=False)

focused_spatial = []
for a, b in family:
    r = spatial_pair(st_med, a, b, "rmse")
    if r is not None:
        focused_spatial.append(r)
focused_spatial = pd.DataFrame(focused_spatial)
focused_spatial["q_bh"] = st.bh_fdr(focused_spatial["t_p"].to_numpy())
focused_spatial.to_csv(EXP_DIR / "spatial_pairwise_focused.csv", index=False)
print("Focused Spatial RMSE comparisons — wins 'k of 5 stations' (A < B), sign test p, paired t p, Wilcoxon p, q = BH-FDR")
print(focused_spatial.sort_values("mean_diff", ascending=True).to_markdown(index=False, floatfmt=".5f"))

Focused Spatial RMSE comparisons — wins 'k of 5 stations' (A < B), sign test p, paired t p, Wilcoxon p, q = BH-FDR
| A                                    | B                                    | metric   |   n_stations |   mean_diff | wins   |   sign_p |     t_p |   wilcoxon_p |    q_bh |
|:-------------------------------------|:-------------------------------------|:---------|-------------:|------------:|:-------|---------:|--------:|-------------:|--------:|
| Trained_Gating_k2_c0_5_c1_10         | Global_Single_54                     | RMSE     |            5 |    -0.01447 | 5 / 5  |  0.06250 | 0.06212 |      0.06250 | 0.39742 |
| Clustering_Backbone54_k2_c0_10_c1_10 | Global_Single_54                     | RMSE     |            5 |    -0.01079 | 4 / 5  |  0.37500 | 0.10497 |      0.12500 | 0.39742 |
| Univariate_G_API_k2_c0_10_c1_0       | Global_Single_54                     | RMSE     |            5 |    -0.00896 | 4 / 5  |  0.37500 | 0.09409 |      0.12500 | 0.39742 |
| Clusteri

# Spatial sample-level cluster bootstrap (5 In-Situ ECE stations)

Paired cluster bootstrap over (station, date) blocks across the 5 ECE stations (150 blocks; seed 42).

In [9]:
"""Sample-level block cluster bootstrap on derived_8.4_ece_v3."""
ece_df = data.ece_all
ece_blocks = (ece_df["station_id"].astype(str) + "_" + ece_df["date"].astype(str)).to_numpy()
y_true_ece = ece_df[data.target].to_numpy(dtype=float)
pred_spatial_dir = EXP_DIR / "predictions_spatial"

boot_spatial_rows = []
for a, b in boot_pairs:
    f_a = pred_spatial_dir / f"{a}__s42__ece_preds.npy"
    f_b = pred_spatial_dir / f"{b}__s42__ece_preds.npy"
    if f_a.exists() and f_b.exists():
        pa = np.load(f_a)
        pb = np.load(f_b)
        res = st.cluster_bootstrap(y_true_ece, pa, pb, ece_blocks, n_resamples=2000, seed=42)
        for m in ("rmse", "r2", "bias"):
            d = res[m]["diff"]
            boot_spatial_rows.append({
                "A": a, "B": b, "metric": m.upper(),
                "diff_mean": d["mean"],
                "diff CI": f"[{d['ci_low']:.5f}, {d['ci_high']:.5f}]",
                "bootstrap_p": d["p"],
            })
boot_spatial_df = pd.DataFrame(boot_spatial_rows)
boot_spatial_df.to_csv(EXP_DIR / "spatial_bootstrap.csv", index=False)
print("Sample-level paired cluster bootstrap over (station, date) blocks on ECE v3 (seed 42):")
print(boot_spatial_df.to_markdown(index=False, floatfmt=".5f"))

Sample-level paired cluster bootstrap over (station, date) blocks on ECE v3 (seed 42):
| A                                    | B                                    | metric   |   diff_mean | diff CI              |   bootstrap_p |
|:-------------------------------------|:-------------------------------------|:---------|------------:|:---------------------|--------------:|
| Clustering_V0_Full_k2_c0_0_c1_10     | Global_Single_54                     | RMSE     |    -0.00220 | [-0.00619, 0.00192]  |       0.27800 |
| Clustering_V0_Full_k2_c0_0_c1_10     | Global_Single_54                     | R2       |     0.11504 | [-0.11195, 0.32052]  |       0.27800 |
| Clustering_V0_Full_k2_c0_0_c1_10     | Global_Single_54                     | BIAS     |     0.01160 | [0.00828, 0.01505]   |       0.00050 |
| Clustering_Backbone54_k2_c0_10_c1_10 | Global_Single_54                     | RMSE     |    -0.00981 | [-0.01363, -0.00600] |       0.00050 |
| Clustering_Backbone54_k2_c0_10_c1_10 | Global_S

# Focused In-Situ ECE Spatial Comparison: Two-Regime (No Deltas) vs Single-Regime Global & Trained Gating

A focused, low-noise comparison evaluating the two-regime models strictly **without regime-specific feature selection**
(identical 54 global backbone features without delta additions, or 50 V0 features for the baseline)
against the single-regime global models and trained-gating models across the 5 in-situ ECE stations (`derived_8.4_ece_v3`).
Ranked primarily by **RMSE** (ascending, lower is better) with Pearson $r$ and $R^2$ presented.

In [10]:
"""Generate focused In-Situ ECE spatial tables (no delta feature selection; ranked by RMSE)."""
target_ece_configs = [
    ("Clustering_Backbone54_k2_c0_0_c1_0", "Clustering (54 backbone)", "Two-Regime (KMeans k=2)"),
    ("Clustering_V0_Full_k2_c0_0_c1_0", "Clustering (50 V0 features)", "Two-Regime (KMeans k=2)"),
    ("Clustering_Dynamic_k2_c0_0_c1_0", "Clustering (Dynamic features)", "Two-Regime (KMeans k=2)"),
    ("Seasonal_Binary_k2_c0_0_c1_0", "Seasonal Binary (Summer/Winter)", "Two-Regime (Heuristic)"),
    ("Univariate_G_API_k2_c0_0_c1_0", "Univariate G_API split", "Two-Regime (Heuristic)"),
    ("Trained_Gating_k2_c0_0_c1_0", "Trained Gating Classifier", "Two-Regime (Supervised Gating)"),
    ("Global_Single_54", "Global Single Model (54 feats)", "Single-Regime (Global)"),
    ("Baseline_V0_50", "Baseline Model (50 V0 feats)", "Single-Regime (Global)"),
]

# 1. Summary Table (Ranked primarily by Station Median RMSE ascending)
rows_summary = []
for cid, label, model_type in target_ece_configs:
    sub_s = spatial[spatial["config_id"] == cid]
    sub_st = st_med[st_med["config_id"] == cid]
    ss_r2 = st.seed_summary(sub_s["r2"])
    ss_rmse = st.seed_summary(sub_s["rmse"])
    ss_mae = st.seed_summary(sub_s["mae"])
    ss_bias = st.seed_summary(sub_s["bias"])
    ss_p = st.seed_summary(sub_s["pearson"]) if "pearson" in sub_s else {"mean": float("nan"), "std": float("nan")}
    
    rows_summary.append({
        "Model Architecture": label,
        "Type": model_type,
        "Station Median RMSE": sub_st["rmse"].median(),
        "Station Mean RMSE": sub_st["rmse"].mean(),
        "Station Median R²": sub_st["r2"].median(),
        "Station Mean R²": sub_st["r2"].mean(),
        "Station Mean Pearson r": sub_st["pearson"].mean() if "pearson" in sub_st else float("nan"),
        "Station Mean MAE": sub_st["mae"].mean(),
        "Station Mean Bias": sub_st["bias"].mean(),
        "Pooled RMSE": f"{ss_rmse['mean']:.4f} ± {ss_rmse['std']:.4f}",
        "Pooled R²": f"{ss_r2['mean']:.4f} ± {ss_r2['std']:.4f}",
        "Pooled Pearson r": f"{ss_p['mean']:.4f}" if not np.isnan(ss_p['mean']) else "—",
    })
df_summary = pd.DataFrame(rows_summary).sort_values("Station Median RMSE", ascending=True)
df_summary.to_csv(EXP_DIR / "spatial_focused_no_delta_summary.csv", index=False)

print("### Table 1: In-Situ ECE Spatial Comparison (5 Unseen Stations, No Delta Selection; Ranked by RMSE)")
print(df_summary.to_markdown(index=False, floatfmt=".4f"))

# 2. Pairwise Hypothesis Testing Table (Head-to-Head on RMSE)
pairs_focused = [
    ("Clustering_V0_Full_k2_c0_0_c1_0", "Global_Single_54", "Clustering (V0) vs Global-54", "Clustering vs Global"),
    ("Clustering_Backbone54_k2_c0_0_c1_0", "Global_Single_54", "Clustering (54) vs Global-54", "Clustering vs Global"),
    ("Clustering_V0_Full_k2_c0_0_c1_0", "Baseline_V0_50", "Clustering (V0) vs Baseline-50", "Clustering vs Baseline"),
    ("Clustering_Backbone54_k2_c0_0_c1_0", "Baseline_V0_50", "Clustering (54) vs Baseline-50", "Clustering vs Baseline"),
    ("Clustering_Dynamic_k2_c0_0_c1_0", "Global_Single_54", "Clustering (Dynamic) vs Global-54", "Clustering vs Global"),
    ("Univariate_G_API_k2_c0_0_c1_0", "Global_Single_54", "Univariate G_API vs Global-54", "Regime vs Global"),
    ("Seasonal_Binary_k2_c0_0_c1_0", "Global_Single_54", "Seasonal Binary vs Global-54", "Seasonal vs Global"),
    ("Clustering_V0_Full_k2_c0_0_c1_0", "Trained_Gating_k2_c0_0_c1_0", "Clustering (V0) vs Trained Gating", "Clustering vs Gating"),
    ("Clustering_Backbone54_k2_c0_0_c1_0", "Trained_Gating_k2_c0_0_c1_0", "Clustering (54) vs Trained Gating", "Clustering vs Gating"),
    ("Clustering_V0_Full_k2_c0_0_c1_0", "Univariate_G_API_k2_c0_0_c1_0", "Clustering (V0) vs Univariate G_API", "Clustering vs Univariate"),
    ("Clustering_V0_Full_k2_c0_0_c1_0", "Clustering_Dynamic_k2_c0_0_c1_0", "Clustering (V0) vs Dynamic Clustering", "Clustering vs Dynamic"),
    ("Clustering_V0_Full_k2_c0_0_c1_0", "Seasonal_Binary_k2_c0_0_c1_0", "Clustering (V0) vs Seasonal Binary", "Clustering vs Seasonal"),
]

rows_pairwise = []
for a, b, label, category in pairs_focused:
    ma = st_med[st_med["config_id"] == a].set_index("station")["rmse"]
    mb = st_med[st_med["config_id"] == b].set_index("station")["rmse"]
    c_st = ma.index.intersection(mb.index)
    s_res = st.station_pair_test(ma.loc[c_st], mb.loc[c_st])
    wins = int(np.sum(ma.loc[c_st] < mb.loc[c_st]))
    
    pa_rmse = spatial[spatial["config_id"] == a]["rmse"].mean()
    pb_rmse = spatial[spatial["config_id"] == b]["rmse"].mean()
    
    pa_r2 = spatial[spatial["config_id"] == a]["r2"].mean()
    pb_r2 = spatial[spatial["config_id"] == b]["r2"].mean()
    
    rows_pairwise.append({
        "Category": category,
        "Comparison (A vs B)": label,
        "Station Mean ΔRMSE (A−B)": s_res["mean_diff"],
        "Station Wins (A < B)": f"{wins} / 5",
        "Binomial Sign Test p": s_res["sign_p"],
        "Paired t-test p": s_res["t_p"],
        "Wilcoxon p": s_res["wilcoxon_p"],
        "Pooled ΔRMSE": pa_rmse - pb_rmse,
        "Pooled ΔR²": pa_r2 - pb_r2,
    })
df_pairwise = pd.DataFrame(rows_pairwise)
df_pairwise.to_csv(EXP_DIR / "spatial_focused_no_delta_pairwise.csv", index=False)

print()
print("### Table 2: Head-to-Head ECE Spatial Pairwise Tests (Per-Station Medians across 5 Stations; ΔRMSE)")
print(df_pairwise.to_markdown(index=False, floatfmt=".5f"))

# 3. Per-Station RMSE Matrix
pivot_focused = st_med[st_med["config_id"].isin([c[0] for c in target_ece_configs])].pivot(
    index="config_id", columns="station", values="rmse"
)
pivot_focused = pivot_focused.reindex([c[0] for c in target_ece_configs])
pivot_focused.index = [c[1] for c in target_ece_configs]
pivot_focused.to_csv(EXP_DIR / "spatial_focused_no_delta_per_station_rmse.csv")

print()
print("### Table 3: Per-Station RMSE Matrix across 5 In-Situ ECE Stations (No Deltas; Lower is Better)")
print(pivot_focused.to_markdown(floatfmt=".4f"))

# 4. Table 4: Cluster Distance & OOD Domain Shift Diagnostics
scaler_54 = StandardScaler()
X_tr_54 = scaler_54.fit_transform(data.trainval[data.shared_backbone_54].fillna(data.trainval[data.shared_backbone_54].mean()))
km_54 = KMeans(n_clusters=2, random_state=42, n_init=10).fit(X_tr_54)
c0_54, c1_54 = km_54.cluster_centers_

d0_tr = np.linalg.norm(X_tr_54 - c0_54, axis=1)
d1_tr = np.linalg.norm(X_tr_54 - c1_54, axis=1)
wa_mean_d = float(np.minimum(d0_tr, d1_tr).mean())
wa_std_d = float(np.minimum(d0_tr, d1_tr).std())

pred_dir = EXP_DIR / "predictions"
f_cl = pred_dir / "Clustering_Backbone54_k2_c0_0_c1_0__s42__full_preds.npy"
p_cl = np.load(f_cl) if f_cl.exists() else None

wa_rows = []
for st_id in sorted(data.test["station_id"].unique()):
    sub = data.test[data.test["station_id"] == st_id]
    X_st = scaler_54.transform(sub[data.shared_backbone_54].fillna(data.trainval[data.shared_backbone_54].mean()))
    d0_st = np.linalg.norm(X_st - c0_54, axis=1)
    d1_st = np.linalg.norm(X_st - c1_54, axis=1)
    d_c = np.minimum(d0_st, d1_st)
    d_s = np.maximum(d0_st, d1_st)
    pred_c = km_54.predict(X_st)
    
    idx = (data.test["station_id"] == st_id).to_numpy()
    y_true = data.test.loc[idx, data.target].to_numpy()
    
    rmse_cl_val = np.sqrt(mean_squared_error(y_true, p_cl[idx])) if p_cl is not None else np.nan
    r2_cl_val = r2_score(y_true, p_cl[idx]) if p_cl is not None else np.nan
    p_corr = float(pearsonr(y_true, p_cl[idx])[0]) if p_cl is not None else np.nan
    
    c0_pct = (pred_c == 0).mean() * 100
    c1_pct = (pred_c == 1).mean() * 100
    
    wa_rows.append({
        "Group": "WA (In-Dist Baseline)",
        "Station": st_id,
        "Clustering RMSE": rmse_cl_val,
        "Clustering R²": r2_cl_val,
        "Pearson r": p_corr,
        "Dist to Closest": float(d_c.mean()),
        "Dist to 2nd Closest": float(d_s.mean()),
        "Margin (2nd − Closest)": float((d_s - d_c).mean()),
        "Ambiguity Ratio": float((d_c / d_s).mean()),
        "OOD Z-Score (vs WA)": float((d_c.mean() - wa_mean_d) / wa_std_d),
        "Cluster Allocation (C0 / C1)": f"{c0_pct:.0f}% / {c1_pct:.0f}%",
        "Target Mean (m³/m³)": float(sub[data.target].mean()),
        "Target Std": float(sub[data.target].std()),
    })

# In-Situ ECE Stations
rmse_backbone_ece = st_med[st_med["config_id"] == "Clustering_Backbone54_k2_c0_0_c1_0"].groupby("station")["rmse"].median()
r2_backbone_ece = st_med[st_med["config_id"] == "Clustering_Backbone54_k2_c0_0_c1_0"].groupby("station")["r2"].median()
p_backbone_ece = st_med[st_med["config_id"] == "Clustering_Backbone54_k2_c0_0_c1_0"].groupby("station")["pearson"].median() if "pearson" in st_med else {}

X_ece_54 = scaler_54.transform(data.ece_all[data.shared_backbone_54].fillna(data.trainval[data.shared_backbone_54].mean()))
d0_ece = np.linalg.norm(X_ece_54 - c0_54, axis=1)
d1_ece = np.linalg.norm(X_ece_54 - c1_54, axis=1)
d_c_ece = np.minimum(d0_ece, d1_ece)
d_s_ece = np.maximum(d0_ece, d1_ece)
pred_c_ece = km_54.predict(X_ece_54)

ece_rows = []
for st_id in sorted(data.ece_all["station_id"].unique()):
    sub_mask = (data.ece_all["station_id"] == st_id).to_numpy()
    sub_df = data.ece_all.iloc[sub_mask]
    d_c = d_c_ece[sub_mask]
    d_s = d_s_ece[sub_mask]
    c0_pct = (pred_c_ece[sub_mask] == 0).mean() * 100
    c1_pct = (pred_c_ece[sub_mask] == 1).mean() * 100
    
    ece_rows.append({
        "Group": "ECE (In-Situ Sensor Transfer)",
        "Station": st_id,
        "Clustering RMSE": float(rmse_backbone_ece.get(st_id, np.nan)),
        "Clustering R²": float(r2_backbone_ece.get(st_id, np.nan)),
        "Pearson r": float(p_backbone_ece.get(st_id, np.nan)),
        "Dist to Closest": float(d_c.mean()),
        "Dist to 2nd Closest": float(d_s.mean()),
        "Margin (2nd − Closest)": float((d_s - d_c).mean()),
        "Ambiguity Ratio": float((d_c / d_s).mean()),
        "OOD Z-Score (vs WA)": float((d_c.mean() - wa_mean_d) / wa_std_d),
        "Cluster Allocation (C0 / C1)": f"{c0_pct:.0f}% / {c1_pct:.0f}%",
        "Target Mean (m³/m³)": float(sub_df[data.target].mean()),
        "Target Std": float(sub_df[data.target].std()),
    })

df_diag_all = pd.concat([pd.DataFrame(wa_rows), pd.DataFrame(ece_rows)], ignore_index=True)
df_diag_all.to_csv(EXP_DIR / "spatial_focused_no_delta_station_cluster_distances.csv", index=False)

print()
print("### Table 4: Station Distance to Clusters & OOD Domain Shift Diagnostics (WA Baseline + 5 ECE Stations)")
print(df_diag_all.to_markdown(index=False, floatfmt=".4f"))

### Table 1: In-Situ ECE Spatial Comparison (5 Unseen Stations, No Delta Selection; Ranked by RMSE)
| Model Architecture              | Type                           |   Station Median RMSE |   Station Mean RMSE |   Station Median R² |   Station Mean R² |   Station Mean Pearson r |   Station Mean MAE |   Station Mean Bias | Pooled RMSE     | Pooled R²        |   Pooled Pearson r |
|:--------------------------------|:-------------------------------|----------------------:|--------------------:|--------------------:|------------------:|-------------------------:|-------------------:|--------------------:|:----------------|:-----------------|-------------------:|
| Trained Gating Classifier       | Two-Regime (Supervised Gating) |                0.0445 |              0.0485 |            -33.0261 |         -198.8126 |                   0.3954 |             0.0463 |              0.0199 | 0.0529 ± 0.0004 | -0.2647 ± 0.0180 |            -0.0545 |
| Baseline Model (50 V0 feats)    | Single-Re


### Table 3: Per-Station RMSE Matrix across 5 In-Situ ECE Stations (No Deltas; Lower is Better)
|                                 |   ECE_BBG_Lost_Meadow |   ECE_BBG_Main_St |   ECE_Renton_Garden_North |   ECE_Renton_Garden_Shed |   ECE_Renton_Home |
|:--------------------------------|----------------------:|------------------:|--------------------------:|-------------------------:|------------------:|
| Clustering (54 backbone)        |                0.0487 |            0.0487 |                    0.0563 |                   0.0354 |            0.0878 |
| Clustering (50 V0 features)     |                0.0487 |            0.0487 |                    0.0563 |                   0.0354 |            0.0878 |
| Clustering (Dynamic features)   |                0.0607 |            0.0452 |                    0.0711 |                   0.0177 |            0.0762 |
| Seasonal Binary (Summer/Winter) |                0.0610 |            0.0446 |                    0.0734 |                   0.


### Table 4: Station Distance to Clusters & OOD Domain Shift Diagnostics (WA Baseline + 5 ECE Stations)
| Group                         | Station                 |   Clustering RMSE |   Clustering R² |   Pearson r |   Dist to Closest |   Dist to 2nd Closest |   Margin (2nd − Closest) |   Ambiguity Ratio |   OOD Z-Score (vs WA) | Cluster Allocation (C0 / C1)   |   Target Mean (m³/m³) |   Target Std |
|:------------------------------|:------------------------|------------------:|----------------:|------------:|------------------:|----------------------:|-------------------------:|------------------:|----------------------:|:-------------------------------|----------------------:|-------------:|
| WA (In-Dist Baseline)         | BeaverPass_WA_990       |            0.0562 |          0.6189 |      0.9375 |            6.0571 |                9.1299 |                   3.0728 |            0.6661 |               -0.1400 | 100% / 0%                      |                0.2345 |       0.0912 

# Figures generation

Exports all publication figures:
1. Spatial seed boxplots (RMSE, R², and Pearson r)
2. Paired differences for headline pairs (RMSE and R²)
3. Spatial pair scatter on 5 ECE stations (RMSE)
4. Spatial station bars with seed error bars
5. Delta-source robustness bars (RMSE and R²)
6. **Chart Suite 1 (Architecture Showdown, NO per-regime deltas):** Chronological line charts ($\le 5$ lines: Observed + `Clustering_V0_Full_k2 c0=0,c1=0`, `Clustering_Backbone54_k2 c0=0,c1=0`, `Global_Single_54`, `Trained_Gating_k2 c0=0,c1=0`).
7. **Chart Suite 2 (Regime Benchmark Showdown):** Chronological line charts ($\le 5$ lines: Observed + 4 models: `Clustering_V0_Full_k2 c0=0, c1=0`, `Univariate_G_API_k2 c0=0, c1=0`, `Clustering_Dynamic_k2 c0=0, c1=0`, `Seasonal_Binary_k2 c0=0, c1=0`).

In [11]:
"""Generate all report figures (enforces <= 5 lines per time series chart)."""
# 1. Boxplots
plots.plot_seed_boxplot(temporal, cfg, EXP_DIR, prefix="temporal", metric="rmse")
plots.plot_seed_boxplot(temporal, cfg, EXP_DIR, prefix="temporal", metric="r2")
plots.plot_seed_boxplot(spatial, cfg, EXP_DIR, prefix="spatial", metric="rmse")
plots.plot_seed_boxplot(spatial, cfg, EXP_DIR, prefix="spatial", metric="r2")
if "pearson" in spatial:
    plots.plot_seed_boxplot(spatial, cfg, EXP_DIR, prefix="spatial", metric="pearson")

# 2. Paired differences for headline pairs
for a, b in plots.HEADLINE_PAIRS:
    plots.plot_paired_differences(temporal, cfg, EXP_DIR, (a, b), prefix="paired_diff_temporal", metric="rmse")
    plots.plot_paired_differences(spatial, cfg, EXP_DIR, (a, b), prefix="paired_diff_spatial", metric="rmse")
    plots.plot_paired_differences(spatial, cfg, EXP_DIR, (a, b), prefix="paired_diff_spatial", metric="r2")

# 3. Spatial pair scatter on 5 ECE stations
for a, b in plots.HEADLINE_PAIRS:
    plots.plot_spatial_pair(st_med, cfg, EXP_DIR, (a, b), metric="rmse")
    plots.plot_spatial_pair(st_med, cfg, EXP_DIR, (a, b), metric="r2")

# 4. Spatial station bars
for cid in ["Clustering_V0_Full_k2_c0_0_c1_10", "Clustering_Backbone54_k2_c0_10_c1_10",
            "Global_Single_54", "Trained_Gating_k2_c0_5_c1_10"]:
    plots.plot_spatial_station_bars(spatial_st, cfg, EXP_DIR, cid, metric="rmse")

# 5. Delta robustness bars
plots.plot_delta_robustness(temporal, spatial, cfg, EXP_DIR, metric="rmse")
plots.plot_delta_robustness(temporal, spatial, cfg, EXP_DIR, metric="r2")

# 6. Chart Suite 1: Architecture Showdown WITHOUT per-regime deltas (Strictly <= 5 lines per chart)
suite1_paths = plots.plot_ece_station_timeseries(
    data.ece_all, config, cfg, EXP_DIR,
    config_ids=plots.ECE_ARCH_NODELTA_CONFIGS,
    suite_suffix="architecture"
)

# 7. Chart Suite 2: Regime Benchmark Showdown (Strictly <= 5 lines per chart)
suite2_paths = plots.plot_ece_station_timeseries(
    data.ece_all, config, cfg, EXP_DIR,
    config_ids=plots.ECE_REGIME_CONFIGS,
    suite_suffix="regime_benchmark"
)

print("[Plots] All figures written successfully to", EXP_DIR)
print(f"[Plots] Architecture Suite: {[p.name for p in suite1_paths]}")
print(f"[Plots] Regime Benchmark Suite: {[p.name for p in suite2_paths]}")

[Plots] All figures written successfully to /scratch/group/p.cis250607.000/MDR-Project/notebooks/experiment/derived_8.4-formal-eval-2.1-ece-v3
[Plots] Architecture Suite: ['spatial_ece_timeseries_architecture_combined.png', 'spatial_ece_timeseries_architecture_ECE_BBG_Lost_Meadow.png', 'spatial_ece_timeseries_architecture_ECE_BBG_Main_St.png', 'spatial_ece_timeseries_architecture_ECE_Renton_Garden_North.png', 'spatial_ece_timeseries_architecture_ECE_Renton_Garden_Shed.png', 'spatial_ece_timeseries_architecture_ECE_Renton_Home.png']
[Plots] Regime Benchmark Suite: ['spatial_ece_timeseries_regime_benchmark_combined.png', 'spatial_ece_timeseries_regime_benchmark_ECE_BBG_Lost_Meadow.png', 'spatial_ece_timeseries_regime_benchmark_ECE_BBG_Main_St.png', 'spatial_ece_timeseries_regime_benchmark_ECE_Renton_Garden_North.png', 'spatial_ece_timeseries_regime_benchmark_ECE_Renton_Garden_Shed.png', 'spatial_ece_timeseries_regime_benchmark_ECE_Renton_Home.png']


# Delta-robustness table

Examines whether the two-regime clustering conclusion survives across delta selection sources
(test-selected vs. validation-selected vs. none) on both temporal and in-situ ECE spatial evaluation.

In [12]:
"""Compute delta-robustness table."""
robust = []
for strategy in cfg["strategy_name"].unique():
    if strategy == "Global_Single":
        continue
    row = {"strategy": strategy}
    for source in ("test", "val", "none"):
        ids = cfg[(cfg["strategy_name"] == strategy) & (cfg["delta_source"] == source)]["config_id"]
        if ids.empty:
            row[f"{source}_config"] = "—"
            continue
        cid = ids.iloc[0]
        sub_t = temporal[temporal["config_id"] == cid]
        s_t_rmse = st.seed_summary(sub_t["rmse"])
        s_t_r2 = st.seed_summary(sub_t["r2"])
        row[f"{source}_config"] = cid
        row[f"{source}_temp_rmse"] = f"{s_t_rmse['mean']:.5f}" if s_t_rmse["n"] else "—"
        row[f"{source}_temp_r2"] = f"{s_t_r2['mean']:.4f}" if s_t_r2["n"] else "—"
        
        ssub = spat_cfg_df[spat_cfg_df["config_id"] == cid]
        row[f"{source}_spat_rmse"] = f"{ssub['spatial_mean_rmse'].iloc[0]:.5f}" if len(ssub) else "—"
        row[f"{source}_spat_r2"] = f"{ssub['spatial_mean_r2'].iloc[0]:.4f}" if len(ssub) else "—"
    robust.append(row)
robust = pd.DataFrame(robust)
robust.to_csv(EXP_DIR / "delta_robustness_summary.csv", index=False)
print("### Delta-Source Robustness Table (Temporal WA vs Spatial ECE; RMSE & R²)")
print(robust.to_markdown(index=False))

### Delta-Source Robustness Table (Temporal WA vs Spatial ECE; RMSE & R²)
| strategy                 | test_config                          |   test_temp_rmse |   test_temp_r2 |   test_spat_rmse |   test_spat_r2 | val_config                          |   val_temp_rmse |   val_temp_r2 |   val_spat_rmse |   val_spat_r2 | none_config                        |   none_temp_rmse |   none_temp_r2 |   none_spat_rmse |   none_spat_r2 |
|:-------------------------|:-------------------------------------|-----------------:|---------------:|-----------------:|---------------:|:------------------------------------|----------------:|--------------:|----------------:|--------------:|:-----------------------------------|-----------------:|---------------:|-----------------:|---------------:|
| Clustering_V0_Full_k2    | Clustering_V0_Full_k2_c0_0_c1_10     |          0.04409 |         0.8126 |          0.05836 |        -0.5388 | Clustering_V0_Full_k2_val_winner    |         0.05243 |        0.7351 |     

# Replication checks

Replication check for seed 42 against deterministic baseline anchors.

In [13]:
"""Replication check cell."""
print("TEMPORAL replication (seed 42 pooled test R2 vs eval-1.1 / eval-1.3 full baseline)")
anchors = config.get("replication", {}).get("temporal_r2", {})
for cid, expected in anchors.items():
    r = temporal[(temporal["config_id"] == cid) & (temporal["seed"] == 42)]
    if r.empty:
        print(f"  {cid}: MISSING"); continue
    got = float(r.iloc[0]["r2"]); diff = abs(got - float(expected))
    print(f"  {cid}: got={got:.6f} expected={expected:.6f} |diff|={diff:.2e} "
          f"[{'OK' if diff < 1e-6 else 'MISMATCH'}]")

TEMPORAL replication (seed 42 pooled test R2 vs eval-1.1 / eval-1.3 full baseline)
  Clustering_V0_Full_k2_c0_0_c1_10: got=0.814960 expected=0.814960 |diff|=1.12e-07 [OK]
  Global_Single_54: got=0.779230 expected=0.779230 |diff|=1.95e-07 [OK]
  Baseline_V0_50: got=0.760447 expected=0.760447 |diff|=3.83e-07 [OK]


# Key Takeaways & Discussion (No-Delta Regimes)

1. **Missingness-Aware Router Salvage Success:**
   By applying the availability gate router fix ($\tau = 0.10$), the model gracefully detects the missing SMAP sensor channels in `derived_8.4_ece_v3` and falls back to the SMAP-free `Univariate_G_API_k2` router. This completely resolves the severe failure mode observed in v2.0 (where predictions defaulted to Cluster 1 wet specialist predictions), reducing spatial RMSE from $\sim 0.167$ to $\sim 0.050\text{ m}^3/\text{m}^3$.
2. **Evaluation Metric Realism (RMSE vs R²):**
   In this 30-day late-summer dry window (July 20 to August 19, 2026), soil moisture variance is extremely small ($\sigma_y \approx 0.003$ to $0.008\text{ m}^3/\text{m}^3$). Small absolute errors ($\text{RMSE} \approx 0.04$ to $0.05\text{ m}^3/\text{m}^3$) unavoidably drive $R^2 = 1 - \text{MSE}/\text{Var}(y)$ negative. Ranking models primarily by **RMSE** provides an uncorrupted, physically grounded assessment of sensor transfer accuracy.
3. **No-Delta Verdict: regime partitioning alone does not transfer on RMSE.**
   Evaluated strictly WITHOUT per-regime feature selection (identical 54 global backbone features, `c0=0,c1=0`; README Tables 1–3 sourced from `spatial_focused_no_delta_*.csv`), the two clustering regime models tie the single-regime global model and lose to the no-delta trained gating and V0 baseline across the 5 ECE stations:
   - Clustering (V0, `c0=0,c1=0`) pooled RMSE $0.0584 \pm 0.0010$ vs Global-Single-54 $0.0586 \pm 0.0007$ (pooled $\Delta$RMSE $-0.00021$); per-station median $\Delta$RMSE $+0.00025$, 2/5 wins, binomial sign $p = 1.0000$, paired t $p = 0.97652$, Wilcoxon $p = 1.00000$ — statistically indistinguishable.
   - Clustering (V0) vs Baseline-50: station mean $\Delta$RMSE $+0.00828$, 1/5 wins (sign $p = 0.37500$, t $p = 0.42408$) — loses.
   - Clustering (V0) vs Trained Gating (`c0=0,c1=0`): station mean $\Delta$RMSE $+0.00688$, 1/5 wins (sign $p = 0.37500$, t $p = 0.27348$) — loses; no-delta Trained Gating holds the best station-median RMSE ($0.0445$), ahead of Baseline-50 ($0.0448$) and Clustering ($0.0487$).
   - The Backbone54 no-delta twin is numerically identical to V0 no-delta on ECE (pooled RMSE $0.0584$ both; per-station RMSE equal to 4dp), so its two lines overlap in the Architecture Showdown figure.
   - No other no-delta regime beats Global-54 either: Univariate G_API 3/5 ($\Delta -0.00083$, sign $p = 1.0$), Dynamic 3/5 ($\Delta -0.00097$, sign $p = 1.0$), Seasonal Binary 2/5 ($\Delta +0.00018$).
   Conclusion: without per-regime delta features, two-regime partitioning provides no spatial RMSE benefit over a single global model on these 5 in-situ stations. Regime gains reported elsewhere in this experiment come from the delta-feature variants, not from partitioning alone.
4. **Trend Directionality (Pearson $r$) is the one preserved strength:**
   Clustering no-delta attains the best station-mean Pearson $r$ ($0.4297$ vs Global-54 $0.4277$, Gating no-delta $0.3954$), so regime models still track dry-down directionality even while RMSE ties — with positive per-station $r$ at 4 of 5 ECE sites (Lost Meadow is negative, $-0.34$).
5. **Why transfer stalls (Table 4 diagnostics):**
   ECE stations sit near the KMeans decision boundary (ambiguity ratio $0.89$–$0.96$ vs $0.56$–$0.72$ on WA; margin $0.26$–$0.88$ vs $2.71$–$4.46$), so hard cluster assignment is near coin-flip on 2 of 5 sites (27%/73% splits at Garden North/Shed). Combined with tiny target variance, small biases dominate RMSE. In-distribution (WA temporal) the same no-delta clustering ranks 2nd–3rd (RMSE $0.04419$–$0.04420$), confirming the failure is spatial transfer, not model quality.

In [14]:
print("Notebook complete — all tables above are the README source of truth.")

Notebook complete — all tables above are the README source of truth.
